# Building the AI Code Reviewer Pipeline

This notebook integrates:

- ChromaDB Retrieval
- Llama 3.2 via Ollama
- Prompt Engineering

The system retrieves relevant coding knowledge and uses it to generate code reviews, identify bugs, suggest fixes, and produce corrected code.

In [1]:
import chromadb
import ollama

In [2]:
client = chromadb.PersistentClient(
    path="../chroma_db"
)

collection = client.get_collection(
    name="code_review_knowledge"
)

print("Connected")

Connected


In [3]:
#Later this will be the code that we want to review, but for now we will just test the connection to Ollama and the model.
user_code = """
def divide(a,b):
    return a/b

print(divide(10,0))
"""

In [4]:
results = collection.query(
    query_texts=[user_code],
    n_results=3
)

context = "\n".join(
    results["documents"][0]
)

print(context)


PYTHON CODING STANDARDS

1. Follow PEP8 naming conventions.
2. Use meaningful variable and function names.
3. Keep functions small and focused on a single task.
4. Add docstrings for important functions and classes.
5. Use type hints where possible.
6. Avoid global variables.
7. Handle exceptions using try-except blocks.
8. Avoid bare except statements.
9. Use list comprehensions when they improve readability.
10. Remove unused imports and variables.
11. Use constants for fixed values.
12. Avoid code duplication.
13. Validate user inputs before processing.
14. Close files using context managers (with statement).
15. Use logging instead of excessive print statements.
16. Prevent division by zero errors.
17. Write modular and reusable code.
18. Follow object-oriented principles when appropriate.
19. Use virtual environments for dependency management.
20. Write unit tests for critical functions.

JAVA CODING STANDARDS

1. Follow SOLID principles.
2. Use meaningful class, method, and vari

In [5]:
prompt = f"""
You are a senior software engineer.

Context:
{context}

Review the following code.

Tasks:

1. Find bugs
2. Mention line numbers
3. Explain issues
4. Suggest fixes
5. Generate corrected code
6. Give a code quality score out of 10

Code:

{user_code}
"""

In [6]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

review = response["message"]["content"]

print(review)

**Bug Report**

1. **Division by Zero**: 
   Issue: The function will attempt to perform division when `b` is zero.
   Line number: 4
   Severity: High
   Fix: Add a condition to check if `b` is zero before performing the division.

   Suggested fix:
   ```python
def divide(a,b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a/b

print(divide(10, 0))  # Raises ValueError: Cannot divide by zero
```

2. **Potential Division Issue**: 
   Issue: The function will not handle cases where `a` is zero.
   Line number: Not specified ( generic issue)
   Severity: Medium
   Fix: Add a condition to check if `a` is zero before performing the division.

   Suggested fix:
   ```python
def divide(a,b):
    if a == 0 or b == 0:
        raise ValueError("Cannot divide by zero")
    return a/b

print(divide(10, 0))  # Raises ValueError: Cannot divide by zero
```

**Code Suggestions**

The provided code seems to be following the general structure of a simple division function